In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
import tensorflow as tf
import numpy as np
import lyrebird as lyrebird


2024-03-17 23:25:10.438756: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-17 23:25:11.228038: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [74]:
input_file = 'input.wav'
output_file = 'bigmuff.wav'
channels = 1
window_size = 200
sample_len = 100000


In [75]:
input_data, output_data = lyrebird.sample_wavs(input_file, output_file, window_size, sample_len, False)

In [119]:
model = tf.keras.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(8, activation='relu'),
  tf.keras.layers.Dense(8, activation='relu'),
  tf.keras.layers.Dense(1)
])

In [120]:
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

In [124]:
print(input_data.shape)
print(output_data.shape)

history = model.fit(
    x = input_data,
    y = output_data,
    validation_split = 0.1,
    shuffle = True,
    epochs=10)

(100000, 200, 1)
(100000, 1)
Epoch 1/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.5405e-04 - mae: 0.0083 - val_loss: 1.5666e-04 - val_mae: 0.0086
Epoch 2/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.5082e-04 - mae: 0.0082 - val_loss: 1.3634e-04 - val_mae: 0.0079
Epoch 3/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.4883e-04 - mae: 0.0081 - val_loss: 1.4378e-04 - val_mae: 0.0081
Epoch 4/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.4589e-04 - mae: 0.0080 - val_loss: 1.4130e-04 - val_mae: 0.0080
Epoch 5/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.4368e-04 - mae: 0.0080 - val_loss: 1.3829e-04 - val_mae: 0.0080
Epoch 6/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.4113e-04 - mae: 0.0079 - val_loss: 1.2482e-04 - val_mae: 0.0074
Epoch 7/10
2813/2813 [==============================] - 6s 2ms/step - loss: 1.4032e-04 - mae: 0.0079 - val_loss: 2.0315

In [ ]:
test_input_samples, test_output_samples = lyrebird.sample_wavs(input_file, output_file, window_size, 44100 * 5, True)

In [125]:
print(test_input_samples.shape)

prediction = model.predict(test_input_samples)

(220500, 200, 1)
6891/6891 [==============================] - 10s 1ms/step


In [126]:
prediction_audio = tf.audio.encode_wav(prediction, 44100)
tf.io.write_file('prediction.wav', prediction_audio)

In [89]:
actual_audio = tf.audio.encode_wav(test_output_samples, 44100)
tf.io.write_file('actual.wav', actual_audio)